In [1]:
try:
    %pip install --user "oracledb" --no-warn-script-location
except Exception as e:
    print("\x1b[31m\u2717 Unexpected error! Please contact course staff\n" +
         "Please include the entire text above and below in your message.")
    raise

Note: you may need to restart the kernel to use updated packages.


In [6]:
import oracledb

#Make sure to put in your user and password before running rest of code!
dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(
    user="ora_<YOUR_CWL>",
    password="a<YOUR_STUDENT_NUM>",
    dsn=dsn
)

cur = connection.cursor()

for table in ["OSCAR_NOMINATIONS", "MOVIE_METRICS", "MOVIES"]:
        cur.execute(f"DROP TABLE {table} PURGE")
        print(f"Dropped {table}")


Dropped OSCAR_NOMINATIONS
Dropped MOVIE_METRICS
Dropped MOVIES


In [8]:
cur.execute("""
CREATE TABLE MOVIES (
    movie_id INTEGER PRIMARY KEY,
    title VARCHAR2(255) NOT NULL,
    releaseDate DATE NOT NULL,
    genre VARCHAR2(255),
    language VARCHAR2(100)
)
""")

cur.execute("""
CREATE TABLE MOVIE_METRICS (
    movie_id INTEGER PRIMARY KEY,
    audienceScore INTEGER NOT NULL,
    tomatoMeter INTEGER NOT NULL,
    popularity INTEGER,
    vote_average INTEGER,
    vote_count INTEGER,
    FOREIGN KEY (movie_id) REFERENCES MOVIES(movie_id)
)
""")

cur.execute("""
CREATE TABLE OSCAR_NOMINATIONS (
    nomination_id INTEGER PRIMARY KEY,
    movie_id INTEGER NOT NULL,
    year_film INTEGER NOT NULL,
    year_ceremony INTEGER NOT NULL,
    ceremony INTEGER NOT NULL,
    canon_category VARCHAR2(255) NOT NULL,
    name VARCHAR2(255),
    winner INTEGER NOT NULL,
    FOREIGN KEY (movie_id) REFERENCES MOVIES(movie_id)
)
""")

connection.commit()
print("Tables created.")

Tables created.


In [9]:
for row in cur.execute("""
    SELECT table_name
    FROM user_tables
    ORDER BY table_name
"""):
    print(row[0])

AUTHORS
EDITORS
MOVIES
MOVIE_METRICS
OSCAR_NOMINATIONS
PUBLISHERS
SALES
SALESDETAILS
TITLEAUTHORS
TITLEDITORS
TITLES


In [16]:
import csv
import os
os.getcwd()


import csv
with open("cleaned_schema/cpsc368_group19/generated/movies_table.csv", "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        cur.execute("""
            INSERT INTO MOVIES (movie_id, title, releaseDate, genre, language)
            VALUES (:1, :2, TO_DATE(:3, 'YYYY-MM-DD'), :4, :5)
        """, (
            int(row["movie_id"]),
            row["title"],
            row["releaseDate"],
            row["genre"] if row["genre"] != "" else None,
            row["language"] if row["language"] != "" else None
        ))

connection.commit()
print("MOVIES inserted.")


MOVIES inserted.


In [21]:
import csv

with open("cleaned_schema/cpsc368_group19/generated/movie_metrics_table.csv", "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        cur.execute("""
            INSERT INTO MOVIE_METRICS
            (movie_id, audienceScore, tomatoMeter, popularity, vote_average, vote_count)
            VALUES (:1, :2, :3, :4, :5, :6)
        """, (
            int(float(row["movie_id"])),
            int(float(row["audienceScore"])) if row["audienceScore"] != "" else None,
            int(float(row["tomatoMeter"])) if row["tomatoMeter"] != "" else None,
            int(float(row["popularity"])) if row["popularity"] != "" else None,
            int(float(row["vote_average"])) if row["vote_average"] != "" else None,
            int(float(row["vote_count"])) if row["vote_count"] != "" else None
        ))

connection.commit()
print("MOVIE_METRICS inserted.")

MOVIE_METRICS inserted.


In [24]:


with open("cleaned_schema/cpsc368_group19/generated/oscar_nominations_table.csv", "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        winner_value = 1 if str(row["winner"]).strip().lower() == "true" else 0

        cur.execute("""
            INSERT INTO OSCAR_NOMINATIONS
            (nomination_id, movie_id, year_film, year_ceremony, ceremony, canon_category, name, winner)
            VALUES (:1, :2, :3, :4, :5, :6, :7, :8)
        """, (
            int(float(row["nomination_id"])),
            int(float(row["movie_id"])),
            int(float(row["year_film"])),
            int(float(row["year_ceremony"])),
            int(float(row["ceremony"])),
            row["canon_category"],
            row["name"] if row["name"] != "" else None,
            winner_value
        ))

connection.commit()
print("OSCAR_NOMINATIONS inserted.")

OSCAR_NOMINATIONS inserted.
